# Influence of homophily and heterophily on oversmoothing of GAT and TransformerConv

In [1]:
import torch
import torch_geometric
from typing import Any
import itertools
from torch_geometric.datasets import HeterophilousGraphDataset
import json
import copy
import os

c:\Users\Mislav.FERIT-PC\Desktop\oversmoothing-GAT-TransformerConv\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] device in use {DEVICE}")

[INFO] device in use cuda


## 1 Datasets

### 1.1 High homophily graphs

#### 1.1.1 pubmed dataset

In [3]:
path_pubmed = "./data/PubMed"

In [4]:
pubmed_dataset = torch_geometric.datasets.Planetoid(
    root=path_pubmed,
    name="PubMed",
    transform=torch_geometric.transforms.NormalizeFeatures()
)

In [5]:
print("Dataset information")
print("===================")
print(f"Number of graphs in dataset: {len(pubmed_dataset)}")
print(f"Number of features: {pubmed_dataset.num_features}")
print(f"Number of classes: {pubmed_dataset.num_classes}")

Dataset information
Number of graphs in dataset: 1
Number of features: 500
Number of classes: 3


In [6]:
pubmed_data = pubmed_dataset[0]
print(pubmed_data)

Data(x=[19717, 500], edge_index=[2, 88648], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717])


In [7]:
print("Graph information")
print("==================")
print(f"Number of nodes: {pubmed_data.num_nodes}")
print(f"Number of edges: {pubmed_data.num_edges}")
print(f"Has isolated nodes: {pubmed_data.has_isolated_nodes()}")
print(f"Has self loops: {pubmed_data.has_self_loops()}")

Graph information
Number of nodes: 19717
Number of edges: 88648
Has isolated nodes: False
Has self loops: False


### 1.2 Low homophily graphs

#### 1.2.1 roman empire

In [8]:
path_empire = "./data/Roman-empire"

dataset_empire = HeterophilousGraphDataset(root=path_empire, name="Roman-empire")

In [9]:
print("Dataset information")
print("=====================")
print(f"Number of graphs in dataset: {len(dataset_empire)}")
print(f"Number of features: {dataset_empire.num_features}")
print(f"Number of classes: {dataset_empire.num_classes}")

Dataset information
Number of graphs in dataset: 1
Number of features: 300
Number of classes: 18


In [10]:
data_empire = dataset_empire[0]
print(data_empire)

Data(x=[22662, 300], edge_index=[2, 65854], y=[22662], train_mask=[22662, 10], val_mask=[22662, 10], test_mask=[22662, 10])


In [11]:
print("Graph information")
print("==================")
print(f"Number of nodes: {pubmed_data.num_nodes}")
print(f"Number of edges: {pubmed_data.num_edges}")
print(f"Has isolated nodes: {pubmed_data.has_isolated_nodes()}")
print(f"Has self loops: {pubmed_data.has_self_loops()}")

Graph information
Number of nodes: 19717
Number of edges: 88648
Has isolated nodes: False
Has self loops: False


## 2 GAT network definition

In [12]:
class GATBlock(torch.nn.Module):
    def __init__(self, in_channels, hidden_size, heads, dropout_rate):
        super().__init__()
        self.conv = torch_geometric.nn.GATv2Conv(
            in_channels, 
            hidden_size, 
            heads=heads, 
            concat=True, 
            dropout=dropout_rate
        )
        
        self.project_back = torch.nn.Linear(hidden_size * heads, in_channels)
        self.dropout = torch.nn.Dropout(dropout_rate)

    def forward(self, x, edge_index):
        x = self.conv(x, edge_index)
        x = self.project_back(x)
        return self.dropout(x)

class GATNetwork(torch.nn.Module):
    def __init__(self, number_of_layers, input_size, hidden_size, output_size, num_heads, dropout_rate):
        super().__init__()
        self.layers = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()

        self.input_proj = torch.nn.Linear(input_size, hidden_size)

        for _ in range(number_of_layers):
            self.norms.append(torch_geometric.nn.LayerNorm(hidden_size))
            self.layers.append(GATBlock(hidden_size, hidden_size, num_heads, dropout_rate))

        self.post_lin = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x, edge_index):
        x = self.input_proj(x)
        
        for norm, layer in zip(self.norms, self.layers):
            identity = x 
            
            x = norm(x)
            x = layer(x, edge_index)
            x = torch.nn.functional.gelu(x)

            x = x + identity
                
        return self.post_lin(x)

## 3 TransformerConv network definition

In [13]:
class TransformerBlock(torch.nn.Module):
    def __init__(self, in_channels, hidden_size, heads, dropout_rate):
        super().__init__()
        self.conv = torch_geometric.nn.conv.TransformerConv(in_channels, hidden_size, heads=heads, concat=True, dropout=dropout_rate)
        
        self.project_back = torch.nn.Linear(hidden_size * heads, in_channels)
        self.dropout = torch.nn.Dropout(dropout_rate)

    def forward(self, x, edge_index):
        x = self.conv(x, edge_index)
        x = self.project_back(x) 
        return self.dropout(x)

class TransformerConvNetwork(torch.nn.Module):
    def __init__(self, number_of_layers, input_size, hidden_size, output_size, num_heads, dropout_rate):
        super().__init__()
        self.layers = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()
        
        self.input_proj = torch.nn.Linear(input_size, hidden_size)

        for _ in range(number_of_layers):
            self.norms.append(torch_geometric.nn.LayerNorm(hidden_size))
            self.layers.append(TransformerBlock(hidden_size, hidden_size, num_heads, dropout_rate))

        self.post_lin = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x, edge_index):
        x = self.input_proj(x)
        
        for norm, layer in zip(self.norms, self.layers):
            identity = x 
            
            x = norm(x)
            x = layer(x, edge_index)
            x = torch.nn.functional.gelu(x)
            
            x = x + identity
                
        return self.post_lin(x)

# 4 Transductive learning setting

In [14]:
class TransductiveTrainer:
    def __init__(
        self,
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        loss_function: torch.nn.Module,
        data: torch_geometric.data.Data,
        epochs: int
    ):
        self.model = model
        self.optimizer = optimizer
        self.loss_function = loss_function
        self.data = data
        self.epochs = epochs
        self.logger = {
            "train_loss": [],
            "train_accuracy": [],
            "val_loss": [],
            "val_accuracy": []
        }
    
    def standardize_splits(self, data):
        for mask_name in ['train_mask', 'val_mask', 'test_mask']:
            if hasattr(data, mask_name):
                mask = getattr(data, mask_name)
                if mask.dim() > 1:
                    setattr(data, mask_name, mask[:, 0])
                if getattr(data, mask_name).dtype != torch.bool:
                    new_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
                    new_mask[mask] = True
                    setattr(data, mask_name, new_mask)
        return data

    def fit(self, patiance: int = 5):

        self.data = self.standardize_splits(self.data)

        best_val_acc = 0.0
        epochs_passed = 0

        for epoch in range(self.epochs):
            # training step
            self.model.train()
            logits = self.model(self.data.x, self.data.edge_index)
            train_loss_value = self.loss_function(logits[self.data.train_mask], self.data.y[self.data.train_mask])
            self.optimizer.zero_grad()
            train_loss_value.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            train_accuracy = self.accuracy(logits[self.data.train_mask].argmax(dim=1), self.data.y[self.data.train_mask])

            train_loss_value = train_loss_value.detach().cpu().numpy()
            train_accuracy = train_accuracy.detach().cpu().numpy()
            print(f"[INFO] epoch {epoch + 1}:\n\ttraining loss: {train_loss_value}")
            print(f"\n\ttraining accuracy: {train_accuracy}")
            self.logger["train_loss"].append(train_loss_value)
            self.logger["train_accuracy"].append(train_accuracy)

            # validation step
            self.model.eval()
            with torch.no_grad():
                logits = self.model(self.data.x, self.data.edge_index)
                val_loss_value = self.loss_function(logits[self.data.val_mask], self.data.y[self.data.val_mask])
                val_accuracy = self.accuracy(logits[self.data.val_mask].argmax(dim=1), self.data.y[self.data.val_mask])
                val_loss_value = val_loss_value.cpu().numpy()
                val_accuracy = float(val_accuracy.cpu().numpy())

                print(f"\n\tval loss: {val_loss_value}")
                print(f"\n\tval accuracy: {val_accuracy}")
                print(f"\n")
                self.logger["val_loss"].append(val_loss_value)
                self.logger["val_accuracy"].append(val_accuracy)

                # early stopping
                if val_accuracy > best_val_acc:
                    best_model_state = copy.deepcopy(self.model.state_dict())
                    best_val_acc = val_accuracy
                    epochs_passed = 0
                else:
                    epochs_passed += 1
                    if epochs_passed >= patiance:
                        break
        
        self.model.load_state_dict(best_model_state)
    
    def accuracy(self, y_predict: torch.Tensor, y_truth: torch.Tensor):
        return torch.sum(y_predict == y_truth) / len(y_truth)

    @torch.no_grad()
    def get_energy(self):
        self.model.eval()
        logits = self.model(self.data.x, self.data.edge_index)
        energy = self.compute_dirichlet_energy(logits, self.data.edge_index)

        return float(energy.cpu().numpy())

    def compute_dirichlet_energy(self, x: torch.Tensor, edge_index: torch.Tensor):
        row, col = edge_index
        # Normalize node features to unit hypersphere to ensure energy 
        # is about angle between nodes and not just raw magnitude.
        x_norm = torch.nn.functional.normalize(x, p=2, dim=-1)
        
        source, target = x_norm[row], x_norm[col]
        squared_diff = torch.pow(source - target, 2).sum(dim=-1)
        
        return squared_diff.mean()
    
    @torch.no_grad()
    def test(self, y_predict: torch.Tensor | None = None, y_truth: torch.Tensor | None = None):
        self.model.eval()
        if y_predict is None and y_truth is None:
            logits = self.model(self.data.x, self.data.edge_index)
            y_predict = logits[self.data.test_mask].argmax(dim=1)
            y_truth = self.data.y[self.data.test_mask]
        test_accuracy = self.accuracy(y_predict, y_truth)

        return float(test_accuracy.cpu().numpy())

# 5 Experiment class

In [15]:
class Experiment:
    def __init__(
        self,
        layers: list[int],
        datasets: list[torch_geometric.data.Dataset],
        epochs: int,
        base_layers: int,
        hyperparameter_grid: dict[str, list[Any]],
        hyperparams_optim_epochs: int,
        hyperparameters_path: str
    ):
        self._hyperparameter_grid = hyperparameter_grid
        self._layers = layers
        self._datasets = datasets
        self._base_layers = base_layers
        self._epochs = epochs
        self._hyperparams_optim_epochs = hyperparams_optim_epochs
        self._hyperparameters_path = hyperparameters_path

        self._best_hyperparams_gat = []
        self._best_hyperparams_transformer = []
        self._best_base_accs_gat = []
        self._best_base_accs_trans = []

        self._log = {}
    
    def perform_experiment(self):
        #1. find best hyperparameters on each dataset for both architectures using smaller models
        # (heads, hidden_dim_per_head) are paired in such a way that hidden dimensionality is always the same
        if os.path.exists(self._hyperparameters_path):
            print("[INFO] Hyperparameters found!")
            with open(self._hyperparameters_path, "r") as file:
                hyperparameters_data = json.load(file)
                self._best_hyperparams_gat = hyperparameters_data["gat"]
                self._best_hyperparams_transformer = hyperparameters_data["trans"]
        else:
            print("[INFO] hyperparameters are being optimized!")
            self.optimize_hyperparameters()

        #2. for each dataset, load best hyperparameters for both architectures
        # iterate over provided layers and train both architectures for each layer count
        for dataset_idx, dataset in enumerate(self._datasets):
            metrics = {
                "gat": {
                    "dirichlet_start": [],
                    "acc": [],
                    "dirichlet": []
                },
                "trans": {
                    "dirichlet_start": [],
                    "acc": [],
                    "dirichlet": []
                }
            }
            for layer_num in self._layers:
                heads_gat, hidden_dim_gat = self._best_hyperparams_gat[dataset_idx]["heads_hidden_dim"]
                gat_trainer = self.get_trainers(
                    number_of_layers = layer_num,
                    input_size = dataset.num_features,
                    hidden_dim = hidden_dim_gat,
                    output_size = dataset.num_classes,
                    heads = heads_gat,
                    dropout = self._best_hyperparams_gat[dataset_idx]["dropout"],
                    lr = self._best_hyperparams_gat[dataset_idx]["lr"],
                    weight_decay = self._best_hyperparams_gat[dataset_idx]["weight_decay"],
                    data = dataset[0],
                    epochs = self._epochs,
                    model_type = "gat"
                )[0]

                heads_trans, hidden_dim_trans = self._best_hyperparams_transformer[dataset_idx]["heads_hidden_dim"]
                trans_trainer = self.get_trainers(
                    number_of_layers = layer_num,
                    input_size = dataset.num_features,
                    hidden_dim = hidden_dim_trans,
                    output_size = dataset.num_classes,
                    heads = heads_trans,
                    dropout = self._best_hyperparams_transformer[dataset_idx]["dropout"],
                    lr = self._best_hyperparams_transformer[dataset_idx]["lr"],
                    weight_decay = self._best_hyperparams_transformer[dataset_idx]["weight_decay"],
                    data = dataset[0],
                    epochs = self._epochs,
                    model_type = "trans"
                )[0]

                gat_dirichlet_start = gat_trainer.get_energy()
                trans_dirichlet_start = trans_trainer.get_energy()
                metrics["gat"]["dirichlet_start"].append(gat_dirichlet_start)
                metrics["trans"]["dirichlet_start"].append(trans_dirichlet_start)

                gat_trainer.fit()
                trans_trainer.fit()
                gat_accuracy = gat_trainer.test()
                trans_accuracy = trans_trainer.test()
                gat_dirichlet = gat_trainer.get_energy()
                trans_dirichlet = trans_trainer.get_energy()

                metrics["gat"]["acc"].append(gat_accuracy)
                metrics["trans"]["acc"].append(trans_accuracy)
                metrics["gat"]["dirichlet"].append(gat_dirichlet)
                metrics["trans"]["dirichlet"].append(trans_dirichlet)
            
            self._log[dataset.name] = metrics

        return self._log

    def optimize_hyperparameters(self):
        for dataset in self._datasets:
            best_gat, best_trans, best_gat_accuracy, best_trans_accuracy = self.grid_search(dataset)
            self._best_hyperparams_gat.append(best_gat)
            self._best_hyperparams_transformer.append(best_trans)
            self._best_base_accs_gat.append(best_gat_accuracy)
            self._best_base_accs_trans.append(best_trans_accuracy)
        
        hyperparams_data = {
            "datasets": [dataset.name for dataset in self._datasets],
            "gat": self._best_hyperparams_gat,
            "trans": self._best_hyperparams_transformer
        }

        with open(self._hyperparameters_path, "w") as file:
            json.dump(hyperparams_data, file)
    
    def get_combination_generator(self):
        keys = self._hyperparameter_grid.keys()
        values = self._hyperparameter_grid.values()
        hyperparams_combinations = itertools.product(*values)
        for combination in hyperparams_combinations:
            yield dict(zip(keys, combination))
    
    def get_trainers(
        self,
        number_of_layers: int,
        input_size: int,
        hidden_dim: int,
        output_size: int,
        heads: int,
        dropout: float,
        lr: float,
        weight_decay: float,
        data: torch_geometric.data.Data,
        epochs: int,
        model_type: str | None = None
    ):
        trainers = []

        if model_type == "gat" or model_type == None:
            gat_base = GATNetwork(
                number_of_layers = number_of_layers,
                input_size = input_size,
                hidden_size = hidden_dim,
                output_size = output_size,
                num_heads = heads,
                dropout_rate = dropout
            ).to(DEVICE)
            optimizer_gat = torch.optim.Adam(
                params = gat_base.parameters(),
                lr = lr,
                weight_decay = weight_decay
            )
            gat_trainer = TransductiveTrainer(
                model = gat_base,
                optimizer = optimizer_gat,
                loss_function = torch.nn.CrossEntropyLoss(),
                data = data,
                epochs = epochs
            )
            trainers.append(gat_trainer)

        if model_type == "trans" or model_type == None:
            trans_base = TransformerConvNetwork(
                number_of_layers = number_of_layers,
                input_size = input_size,
                hidden_size = hidden_dim,
                output_size = output_size,
                num_heads = heads,
                dropout_rate = dropout
            ).to(DEVICE)
            optimizer_trans = torch.optim.Adam(
                params = trans_base.parameters(),
                lr = lr,
                weight_decay = weight_decay
            )
            trans_trainer = TransductiveTrainer(
                model = trans_base,
                optimizer = optimizer_trans,
                loss_function = torch.nn.CrossEntropyLoss(),
                data = data,
                epochs = epochs
            )
            trainers.append(trans_trainer)

        return trainers   

    def grid_search(
        self,
        dataset: torch_geometric.data.Dataset
    ):
        hyperparameter_generator = self.get_combination_generator()

        best_gat_accuracy = 0.0
        best_trans_accuracy = 0.0
        best_gat = None
        best_trans = None

        for hyperparameters in hyperparameter_generator:
            heads, hidden_dim = hyperparameters["heads_hidden_dim"]
            
            gat_trainer, trans_trainer = self.get_trainers(
                number_of_layers = self._base_layers,
                input_size = dataset.num_features,
                hidden_dim = hidden_dim,
                output_size = dataset.num_classes,
                heads = heads,
                dropout = hyperparameters["dropout"],
                lr = hyperparameters["lr"],
                weight_decay = hyperparameters["weight_decay"],
                data = dataset[0],
                epochs = self._hyperparams_optim_epochs
            )

            gat_trainer.fit()
            trans_trainer.fit()
            gat_accuracy = gat_trainer.test()
            trans_accuracy = trans_trainer.test()
            if gat_accuracy > best_gat_accuracy:
                best_gat_accuracy = gat_accuracy
                best_gat = hyperparameters
            if trans_accuracy > best_trans_accuracy:
                best_trans_accuracy = trans_accuracy
                best_trans = hyperparameters
        
        return best_gat, best_trans, best_gat_accuracy, best_trans_accuracy

In [16]:
experiment = Experiment(
    layers = [2, 4, 6, 8, 10, 12, 14, 16, 18],
    datasets = [dataset_empire.to(DEVICE), pubmed_dataset.to(DEVICE)],
    epochs = 100,
    base_layers = 4,
    hyperparameter_grid = {
        "heads_hidden_dim": [(4, 32), (2, 64), (8, 16)],
        "dropout": [0.1, 0.2, 0.3, 0.4],
        "lr": [1e-2, 5e-2, 1e-3, 5e-3, 1e-4],
        "weight_decay": [5e-6, 5e-5, 5e-4]
    },
    hyperparams_optim_epochs = 20,
    hyperparameters_path="./best_hyperparameters.json"
)

In [ ]:
log = experiment.perform_experiment()

In [ ]:
print(log)

{'roman_empire': {'gat': {'dirichlet_start': [0.20965996384620667, 0.0564628541469574, 0.1021483987569809, 0.0479879304766655, 0.03167028725147247, 0.03189564123749733, 0.03435516729950905, 0.028108717873692513, 0.12083178758621216], 'acc': [0.7617366313934326, 0.7910342216491699, 0.7956230044364929, 0.7802682518959045, 0.781680166721344, 0.7726790904998779, 0.7788563370704651, 0.770031750202179, 0.13942816853523254], 'dirichlet': [1.3926925659179688, 1.3811306953430176, 1.2309504747390747, 1.2933785915374756, 1.380536675453186, 1.2934420108795166, 1.1527436971664429, 1.2276571989059448, 0.0036901526618748903]}, 'trans': {'dirichlet_start': [0.45451590418815613, 0.5094842910766602, 0.478613018989563, 0.711867094039917, 0.38274019956588745, 0.15151479840278625, 0.5145658254623413, 0.5553287863731384, 0.6385062336921692], 'acc': [0.772502601146698, 0.7871513962745667, 0.8146840333938599, 0.8224496841430664, 0.8185668587684631, 0.8224496841430664, 0.8270384669303894, 0.8355100154876709, 0